<a href="https://colab.research.google.com/github/MohamadHusseinIsmail/Miscalibration-in-healthcare/blob/main/Miscalibration_in_Healthcare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import seaborn as sns
df = pd.read_csv("/content/dataset.csv")
df.head()
df.info()
df.shape
df.isnull().mean().sort_values(ascending=False).head(10)

In [ ]:
df["condition"].value_counts()
df.groupby("condition")["participant_id"].nunique()

In [ ]:
df.describe()
df.groupby("condition")[["confidence_0_100","correct_binary",
                         "calibration_gap_conf_minus_correct",
                         "checked_sources_binary"]].mean()

In [ ]:

# Confidence distribution

sns.histplot(data=df, x="confidence_0_100", hue="condition", bins=30, kde=True)
plt.show()

In [ ]:
#Calibration Gap

sns.histplot(data=df, x="calibration_gap_conf_minus_correct", hue="condition", bins=30)
plt.show()

In [ ]:
# Correctness rate

sns.barplot(data=df, x="condition", y="correct_binary")
plt.show()

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
corr = df[num_cols].corr()

plt.figure(figsize=(12,8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.show()

In [ ]:
# conidence vs correctness ( calibration Scatter)

sns.scatterplot(
    data=df.sample(2000),
    x="confidence_prob_0_1",
    y="correct_binary",
    hue="condition",
    alpha=0.4
)
plt.show()

In [ ]:
#Does Ai reduce monitoring

sns.boxplot(data=df, x="condition", y="verification_time_sec")
plt.show()

sns.barplot(data=df, x="condition", y="checked_sources_binary")
plt.show()

In [ ]:
# Difficulty vs Condition Interaction


sns.lineplot(
    data=df,
    x="difficulty_z",
    y="calibration_gap_conf_minus_correct",
    hue="condition"
)
plt.show()

In [ ]:
#Does skepticism buffer against miscalibration


sns.scatterplot(
    data=df.sample(2000),
    x="skepticism_z",
    y="calibration_gap_conf_minus_correct",
    hue="condition",
    alpha=0.4
)
plt.show()

In [ ]:
#Does Assertiveness inflate confidence?

sns.scatterplot(
    data=df[df["condition"]!="NoAI"],
    x="ai_assertiveness_0_1",
    y="confidence_0_100",
    hue="condition",
    alpha=0.4
)
plt.show()

In [ ]:
Does Ai increase confidence faster than correctnes?

import statsmodels.formula.api as smf

model_conf = smf.ols(
    "confidence_0_100 ~ C(condition) + difficulty_z + ambiguity_z",
    data=df
).fit()

model_acc = smf.logit(
    "correct_binary ~ C(condition) + difficulty_z + ambiguity_z",
    data=df
).fit()

print(model_conf.summary())
print(model_acc.summary())

In [ ]:
#Miscalibration ( primary outcome)

model_gap = smf.ols(
    "calibration_gap_conf_minus_correct ~ C(condition) + difficulty_z + ambiguity_z",
    data=df
).fit()

print(model_gap.summary())

In [ ]:
#Mediation ( Ai__>Verification__>Miscalibrtion)

model_verify = smf.logit(
    "checked_sources_binary ~ C(condition) + difficulty_z + ambiguity_z",
    data=df
).fit()
print(model_verify.summary())

In [ ]:
model_gap_med = smf.ols(
    "calibration_gap_conf_minus_correct ~ C(condition) + checked_sources_binary + difficulty_z + ambiguity_z",
    data=df
).fit()

print(model_gap_med.summary())

In [ ]:
#Deviation above diagonal= overconfidence

import numpy as np
import matplotlib.pyplot as plt

bins = np.linspace(0,1,11)
df["conf_bin"] = pd.cut(df["confidence_prob_0_1"], bins=bins)

cal = df.groupby(["condition","conf_bin"]).agg(
    avg_conf=("confidence_prob_0_1","mean"),
    emp_acc=("correct_binary","mean")
).reset_index()

for cond in df["condition"].unique():
    sub = cal[cal["condition"]==cond]
    plt.plot(sub["avg_conf"], sub["emp_acc"], marker="o", label=cond)

plt.plot([0,1],[0,1], linestyle="--")
plt.xlabel("Average Confidence")
plt.ylabel("Empirical Accuracy")
plt.legend()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

X = df[[
    "condition",
    "difficulty_z",
    "ambiguity_z",
    "confidence_0_100",
    "checked_sources_binary",
    "ai_readability_0_100",
    "ai_assertiveness_0_1",
    "ai_citations_count",
    "ability_z",
    "skepticism_z"
]]

y = df["correct_binary"]

# One-hot encode condition
X = pd.get_dummies(X, columns=["condition"], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

pred = model.predict_proba(X_test)[:,1]

print("ROC AUC:", roc_auc_score(y_test, pred))
print(classification_report(y_test, model.predict(X_test)))

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,mean_squared_error
import numpy as np


X = df[[
    "condition",
    "difficulty_z",
    "ambiguity_z",
    "checked_sources_binary",
    "verification_time_sec",
    "ai_readability_0_100",
    "ai_assertiveness_0_1",
    "ai_citations_count",
    "ability_z",
    "skepticism_z"
]]

y = df["calibration_gap_conf_minus_correct"]

X = pd.get_dummies(X, columns=["condition"], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

reg = RandomForestRegressor(n_estimators=200, random_state=42)
reg.fit(X_train, y_train)

pred = reg.predict(X_test)

print("R2:", r2_score(y_test, pred))

rmse = np.sqrt(mean_squared_error(y_test, pred))
print("RMSE:", rmse)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, roc_auc_score

df = pd.read_csv("dataset.csv")

# Target
y = df["correct_binary"]

# Features (theoretically meaningful predictors)
X = df[[
    "condition",
    "difficulty_z",
    "ambiguity_z",
    "confidence_0_100",
    "checked_sources_binary",
    "verification_time_sec",
    "ai_readability_0_100",
    "ai_assertiveness_0_1",
    "ai_citations_count",
    "ability_z",
    "skepticism_z"
]]

# Handle categorical variable
X = pd.get_dummies(X, columns=["condition"], drop_first=True)

# Replace NaNs (AI fields are NaN in NoAI)
X = X.fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("ROC AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

In [ ]:
!pip install econml

In [ ]:
from econml.dml import LinearDML
from sklearn.ensemble import RandomForestRegressor

treatment = (df["condition"] == "EarlyAI").astype(int)
Y = df["calibration_gap_conf_minus_correct"]
X = df[["difficulty_z","ambiguity_z","ability_z","skepticism_z"]]

est = LinearDML(model_y=RandomForestRegressor(),
                model_t=RandomForestRegressor(),
                random_state=42)

est.fit(Y, treatment, X=X)
print("Estimated ATE:", est.ate(X))

In [ ]:
import statsmodels.formula.api as smf

df["EarlyAI"] = (df["condition"] == "EarlyAI").astype(int)

model = smf.ols(
    "calibration_gap_conf_minus_correct ~ EarlyAI + difficulty_z + ambiguity_z + ability_z + skepticism_z",
    data=df
).fit()

print(model.summary())